# Pipeline Thu thập và Xử lý Dữ liệu News & Wiki (Full Data)

Quá trình này bao gồm 4 phần chính:
1. **Tải dữ liệu News (BKAI)** từ HuggingFace trực tiếp thành dạng Parquet (không giới hạn số lượng raw).
2. **Mount Google Drive** để lưu dữ liệu lâu dài.
3. **Crawl & Xử lý Wikipedia tiếng Việt**: Gọi MediaWiki API để tải toàn bộ bài viết mới nhất và làm sạch bằng các script giữ nguyên logic.
4. **Deduplicate (Loại bỏ trùng lặp)**: Áp dụng exact-match deduplication cho toàn bộ dữ liệu ở bước 1 và 3.

---
**Thứ tự chạy notebook:**
- Chạy **Phần 0** (cài đặt & import) trước tiên.
- Sau đó chạy lần lượt từng phần theo đúng thứ tự.

## Phần 0: Cài đặt thư viện & Import

Cell này cần được chạy đầu tiên trước tất cả các bước còn lại.

**Thư viện sử dụng:**
- `loguru` — Logging đẹp và dễ dùng.
- `datasets` — Tải dataset từ HuggingFace Hub.
- `requests` — Gọi MediaWiki API cho Wikipedia.
- `pyarrow` — Đọc/ghi file Parquet hiệu quả.
- `tqdm` — Hiển thị progress bar.
- `transformers` — Dependency của `datasets`.

In [ ]:
!pip install loguru datasets requests pyarrow tqdm transformers

In [ ]:
!pip install \
loguru \
datasets \
requests \
pyarrow \
tqdm \
transformers

### Import thư viện toàn cục

Các import dùng chung cho toàn bộ notebook được khai báo tập trung ở đây để tránh xung đột và dễ theo dõi dependency.

In [ ]:
# === Standard Library ===
import hashlib
import html as _html
import json
import os
import re
import time
import unicodedata
import urllib.parse
from pathlib import Path

# === Third-party ===
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from datasets import Dataset, load_dataset
from loguru import logger
from tqdm.auto import tqdm

### Mount Google Drive

Mount Drive để lưu trữ dữ liệu lâu dài giữa các session Colab (tránh mất data khi runtime reset).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

---
## Phần 1: Tải dữ liệu News (`download_datasets.py`)

Tải corpus tin tức BKAI từ `bkai-foundation-models/BKAINewsCorpus` và chuyển thành dạng Parquet.

**Các bước thực hiện:**
- Định nghĩa thư mục đầu ra `data/train/`.
- Hàm `format_size()`: Định dạng kích thước file (B/KB/MB/GB).
- Hàm `download_and_save_dataset()`: Tải dataset từ HuggingFace, tùy chọn giới hạn số dòng, lưu ra Parquet.
- Gọi hàm để tải toàn bộ BKAI News Corpus (2M dòng đầu).

In [ ]:
# --- Cấu hình ---
OUTPUT_DIR = Path("data/train")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
def format_size(num_bytes: int) -> str:
    """Format byte count as human-readable string (B/KB/MB/GB/TB)."""
    for unit in ["B", "KB", "MB", "GB"]:
        if num_bytes < 1024.0:
            return f"{num_bytes:.2f} {unit}"
        num_bytes /= 1024.0
    return f"{num_bytes:.2f} TB"


def download_and_save_dataset(
    name: str,
    split: str,
    output_filename: str,
    max_rows: int | None = None,
) -> None:
    logger.info("Downloading: {}", name)
    ds = load_dataset(name, split=split)

    if max_rows and len(ds) > max_rows:
        ds = ds.select(range(max_rows))

    output_path = OUTPUT_DIR / output_filename
    ds.to_parquet(output_path)

    logger.info(
        "Saved {} rows → {} ({})",
        f"{len(ds):,}",
        output_path,
        format_size(os.path.getsize(output_path)),
    )

In [ ]:
download_and_save_dataset(
    name="bkai-foundation-models/BKAINewsCorpus",
    split="train",
    output_filename="bkai_train.parquet",
    max_rows=2_000_000,
)


---
## Phần 2: Crawl toàn bộ Wikipedia tiếng Việt (`crawl_vi_wiki.py`)

Gọi trực tiếp MediaWiki API để tải toàn bộ bài viết Wikipedia tiếng Việt.

**Các bước thực hiện:**
- **Cấu hình**: API endpoint, User-Agent, batch size.
- **Tạo HTTP session** với retry logic (xử lý 429, maxlag, lỗi mạng).
- **Checkpoint**: Tự động lưu tiến trình, hỗ trợ `resume=True` để tiếp tục khi bị gián đoạn.
- **Crawl**: Duyệt qua tất cả trang wiki theo `allpages` API, tải nội dung từng batch.
- Data được checkpoint liên tục dựa theo mã tiếp tục (`apcontinue`). Set `limit = None`.

In [ ]:
# --- Cấu hình Wikipedia Crawler ---
API_ENDPOINT = "https://vi.wikipedia.org/w/api.php"
USER_AGENT = (
    "vi-wiki-crawler/1.0 "
    "(https://github.com/example/vi-wiki-crawler; contact@example.com) "
    "python-requests/2.x"
)
BATCH_SIZE = 50
CONTENT_BATCH = 50

In [ ]:
# --- Khởi tạo Session & Gọi API với retry logic ---

def make_session() -> requests.Session:
    """Tạo HTTP session với User-Agent chuẩn."""
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})
    return session


def _parse_retry_after(value: str | None, default: int) -> int:
    """Parse header Retry-After thành số giây chờ."""
    if value is None: return default
    try: return int(value)
    except ValueError: return default


def api_get(session: requests.Session, params: dict, retries: int = 5) -> dict:
    """Gọi MediaWiki API GET với retry logic (xử lý 429, maxlag, lỗi JSON)."""
    params.setdefault("format", "json")
    params.setdefault("formatversion", "2")
    for attempt in range(1, retries + 1):
        try:
            resp = session.get(API_ENDPOINT, params=params, timeout=30)
            if resp.status_code == 429:
                if attempt == retries: break
                wait = _parse_retry_after(resp.headers.get("Retry-After"), default=60)
                logger.warning("429 Too Many Requests – waiting {} s (attempt {}/{})", wait, attempt, retries)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            try: data = resp.json()
            except ValueError as exc:
                if attempt < retries: time.sleep(2 ** attempt); continue
                raise RuntimeError("API returned invalid JSON after all retries") from exc
            if "error" in data:
                code = data["error"].get("code", "unknown")
                if code == "maxlag":
                    if attempt == retries: break
                    wait = _parse_retry_after(resp.headers.get("Retry-After"), default=5)
                    time.sleep(wait)
                    continue
                raise RuntimeError(f"API error [{code}]: {data['error'].get('info', '')}")
            return data
        except requests.RequestException as exc:
            if attempt < retries: time.sleep(2 ** attempt)
            else: raise RuntimeError(f"API request failed after {retries} attempts: {exc}") from exc
    raise RuntimeError(f"API request failed after {retries} attempts (rate-limit/maxlag)")

In [ ]:
# --- Checkpoint & Fetch nội dung trang ---

def load_checkpoint(path: Path) -> tuple[dict, bool]:
    """Tải checkpoint từ file JSON. Trả về (data, is_valid)."""
    if path.exists():
        try:
            with path.open("r", encoding="utf-8") as f: return json.load(f), True
        except json.JSONDecodeError: return {}, False
    return {}, True


def save_checkpoint(path: Path, data: dict) -> None:
    """Lưu checkpoint an toàn qua file tạm (.tmp) để tránh corrupt."""
    tmp = path.with_suffix(".json.tmp")
    with tmp.open("w", encoding="utf-8") as f: json.dump(data, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def fetch_page_contents(session: requests.Session, page_ids: list[int], delay: float) -> dict[int, dict]:
    if not page_ids: return {}
    results = {}
    params = {
        "action": "query", "pageids": "|".join(str(pid) for pid in page_ids),
        "prop": "revisions", "rvprop": "content", "rvslots": "main", "maxlag": 5,
    }
    data = api_get(session, params)
    for page in data["query"]["pages"]:
        page_id = page["pageid"]
        title = page.get("title", "")
        revisions = page.get("revisions", [])
        if not revisions: continue
        content = revisions[0].get("slots", {}).get("main", {}).get("content", "")
        results[page_id] = {"title": title, "content": content}
    time.sleep(delay)
    return results

In [ ]:
def crawl(output_dir: Path, max_articles: int | None, delay: float, resume: bool) -> None:
    output_dir.mkdir(parents=True, exist_ok=True)
    output_file = output_dir / "vi_wiki_articles.jsonl"
    checkpoint_file = output_dir / "checkpoint.json"

    if resume:
        checkpoint, checkpoint_valid = load_checkpoint(checkpoint_file)
        if not checkpoint_valid and output_file.exists():
            logger.error("Checkpoint is corrupt but output file exists. Fix manually.")
            return
    else: checkpoint = {}

    seen_ids = set(checkpoint.get("seen_ids", []))
    article_count = checkpoint.get("article_count", 0)
    ap_continue = checkpoint.get("ap_continue")

    logger.info("Starting crawl. Output: {} | Max articles: {} | Resume: {}", output_file, max_articles or "unlimited", resume)
    session = make_session()

    allpages_params = {
        "action": "query", "list": "allpages", "apnamespace": 0,
        "apfilterredir": "nonredirects", "aplimit": BATCH_SIZE, "maxlag": 5,
        "format": "json", "formatversion": "2",
    }
    if ap_continue:
        allpages_params["apcontinue"] = ap_continue
        allpages_params["continue"] = "-||"

    open_mode = "a" if (resume and output_file.exists()) else "w"
    with output_file.open(open_mode, encoding="utf-8") as out_f:
        while True:
            if max_articles and article_count >= max_articles: break

            batch_start_apcontinue = allpages_params.get("apcontinue")
            data = api_get(session, allpages_params.copy())
            pages_meta = data["query"]["allpages"]
            has_more = "continue" in data
            next_ap_continue = data.get("continue", {}).get("apcontinue")

            new_page_ids = [p["pageid"] for p in pages_meta if p["pageid"] not in seen_ids]
            reached_limit = False

            if new_page_ids:
                for i in range(0, len(new_page_ids), CONTENT_BATCH):
                    if reached_limit: break
                    batch_ids = new_page_ids[i : i + CONTENT_BATCH]
                    contents = fetch_page_contents(session, batch_ids, delay)

                    for pid, article in contents.items():
                        if not article["content"]: continue
                        record = {
                            "id": pid, "title": article["title"], "text": article["content"],
                            "url": "https://vi.wikipedia.org/wiki/" + urllib.parse.quote(article["title"].replace(" ", "_"), safe="/:")
                        }
                        out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                        seen_ids.add(pid)
                        article_count += 1

                        if article_count % 100 == 0: logger.info("Crawled {} articles so far...", article_count)
                        if max_articles and article_count >= max_articles: reached_limit = True; break

                    out_f.flush()
                    save_checkpoint(checkpoint_file, {"seen_ids": list(seen_ids), "article_count": article_count, "ap_continue": batch_start_apcontinue})

            if reached_limit: break
            save_checkpoint(checkpoint_file, {"seen_ids": list(seen_ids), "article_count": article_count, "ap_continue": next_ap_continue})
            if not has_more: break
            allpages_params.update(data["continue"])
            time.sleep(delay)


WIKI_RAW_DIR = Path("data/raws")
crawl(output_dir=WIKI_RAW_DIR, max_articles=None, delay=1.0, resume=True)


---
## Phần 3: Làm sạch Wikitext (`process_vi_wiki.py`)

Mở file raw của Wikipedia JSONL bên trên và tiến hành làm sạch toàn bộ theo Regex y nguyên gốc, xuất ra clean JSONL rồi đưa về dạng Dataset (Parquet).

**Quy trình làm sạch:**
- Xóa HTML comments, thẻ `<ref>`, và các HTML tags khác.
- Xóa bảng wiki (`{|...|}`), template (`{{...}}`), và image/file links.
- Giữ lại text trong các wikilink nội bộ (`[[...]]`).
- Chuyển headers `==...==` thành text thuần, xóa list prefix (`*`, `#`, `;`, `:`).
- Cắt bỏ các section cuối thường là tham khảo/liên kết ngoài.
- Chuẩn hóa khoảng trắng và dòng trống.

In [ ]:
TAGS = [
    "div", "span", "small", "big", "b", "i", "u", "s", "br", "p",
    "table", "tr", "th", "td",
    "ul", "ol", "li",
    "dl", "dt", "dd",
    "sub", "sup", "blockquote",
    "nowiki", "code", "pre",
    "gallery", "imagemap", "timeline", "score",
    "syntaxhighlight", "source", "poem",
    "section", "indicator", "templatestyles",
    "hiero", "math", "chem"
]

TAG_PATTERN = rf"<({'|'.join(TAGS)})[^>]*>"

_PATTERNS = [
    (re.compile(r"<!--.*?-->", re.DOTALL), ""),
    (re.compile(r"<ref[^>]*/\s*>", re.IGNORECASE), ""),
    (re.compile(r"<ref[^>]*>.*?</ref>", re.DOTALL | re.IGNORECASE), ""),
    (re.compile(TAG_PATTERN, re.IGNORECASE), " "),
    (re.compile(r"</[a-z]+>", re.IGNORECASE), " "),
    (re.compile(r"<[^>]+>"), ""),
]

_MARKUP = re.compile(r"'''|''|----+")
_MAGIC = re.compile(r"__[A-Z_]+__")
_WHITESPACE = re.compile(r"[ \t]+")
_NEWLINES   = re.compile(r"\n{3,}")
_HEADERS = re.compile(r"^=+\s*(.+?)\s*=+\s*$", re.MULTILINE)
_EMPTY_HEADERS = re.compile(r"^=+\s*=+\s*$", re.MULTILINE)
_LIST_PREFIX = re.compile(r"^([*#;:]+)\s?", re.MULTILINE)
_TERMINAL_SECTION_RE = re.compile(
    r"^\s*=+\s*("
    r"Tham\s+kh\u1ea3o|Ch\u00fa\s+th\u00edch|Ghi\s+ch\u00fa|Li\u00ean\s+k\u1ebft\s+ngo\u00e0i"
    r"|Xem\s+th\u00eam|Th\u01b0\s+m\u1ee5c|\u0110\u1ecdc\s+th\u00eam|Ngu\u1ed3n\s+tham\s+kh\u1ea3o"
    r"|Ch\u00fa\s+gi\u1ea3i|Ghi\s+ch\u00fa\s+v\u00e0\s+tham\s+kh\u1ea3o"
    r"|References?|External\s+links?|See\s+also|Notes?|Bibliography|Further\s+reading"
    r")\s*=+\s*$",
    re.IGNORECASE | re.MULTILINE,
)

In [ ]:
def _remove_balanced_braces(text: str) -> str:
    result = []; depth = 0; i = 0; n = len(text)
    while i < n:
        if text[i:i+2] == "{{": depth += 1; i += 2
        elif text[i:i+2] == "}}":
            if depth > 0: depth -= 1
            i += 2
        else:
            if depth == 0: result.append(text[i])
            i += 1
    return "".join(result)


def _remove_wiki_tables(text: str) -> str:
    result = []; depth = 0; i = 0; n = len(text)
    while i < n:
        if text[i:i+2] == "{|": depth += 1; i += 2
        elif text[i:i+2] == "|}":
            if depth > 0: depth -= 1; i += 2
            else: result.append(text[i]); i += 1
        else:
            if depth == 0: result.append(text[i])
            i += 1
    return "".join(result)

In [ ]:
def _remove_balanced_brackets(text: str) -> str:
    """Xử lý [[...]] links: giữ display text, xóa File/Image/Category."""
    result = []; i = 0; n = len(text)
    while i < n:
        if text[i:i+2] == "[[":
            depth = 1; j = i + 2
            while j < n and depth > 0:
                if text[j:j+2] == "[[": depth += 1; j += 2
                elif text[j:j+2] == "]]": depth -= 1; j += 2
                else: j += 1
            inner = text[i+2:j-2]
            lower = inner.lower().replace("_", " ")
            if any(lower.startswith(p) for p in (
                "file:", "image:", "t\u1eadp tin:", "h\u00ecnh:", "\u1ea3nh:", "category:", "th\u1ec3 lo\u1ea1i:", "media:"
            )):
                i = j; continue
            parts = inner.split("|", 1)
            result.append(parts[-1]); i = j
        else:
            result.append(text[i]); i += 1
    return "".join(result)


def _remove_single_brackets(text: str) -> str:
    text = re.sub(r"\[([a-z][a-z0-9_-]*):([^\]\|]*)\|([^\]]+)\]", r"\3", text)
    text = re.sub(r"\[[a-z]{2}:[^\]]+\]", "", text)
    text = re.sub(r"\[[a-z][a-z0-9_-]*:([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\s\]]+\s+([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\]]+\]", "", text)
    return text

In [ ]:
# --- Pipeline làm sạch chính & xử lý file ---

def _strip_list_prefixes(text: str) -> str:
    """Xóa ký tự list prefix (*, #, ;, :) ở đầu dòng."""
    lines = text.splitlines(); out = []
    for line in lines:
        stripped = line.lstrip()
        m = _LIST_PREFIX.match(stripped)
        if m:
            prefix = m.group(1); content = stripped[m.end():]
            if ";" in prefix and ": " in content:
                term, _, defn = content.partition(": ")
                content = term.strip()
                if defn.strip(): content += ": " + defn.strip()
            content = re.sub(r"^[\s:]+", "", content)
            out.append(content)
        else: out.append(line)
    return "\n".join(out)


def clean_wikitext(raw: str) -> str:
    """Làm sạch hoàn chỉnh một chuỗi wikitext thành plain text."""
    text = raw
    for pattern, repl in _PATTERNS: text = pattern.sub(repl, text)
    text = _remove_wiki_tables(text)
    text = _remove_balanced_braces(text)
    text = re.sub(r"^\s*\|[-+|].*\n?", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*\| [^\n]*\n?", "", text, flags=re.MULTILINE)
    text = _remove_balanced_brackets(text)
    text = _remove_single_brackets(text)
    text = _HEADERS.sub(r"\1", text)
    text = _EMPTY_HEADERS.sub("", text)
    text = _MARKUP.sub("", text)
    text = _MAGIC.sub("", text)
    text = _html.unescape(text)
    text = text.replace("\u00a0", " ")
    text = _strip_list_prefixes(text)

    m = _TERMINAL_SECTION_RE.search(text)
    if m: text = text[:m.start()].rstrip()

    text = _WHITESPACE.sub(" ", text)
    text = _NEWLINES.sub("\n\n", text)
    lines = [line.strip() for line in text.splitlines()]
    return "\n".join(lines).strip()


def process(input_path: Path, output_path: Path):
    """Đọc JSONL raw, áp dụng clean_wikitext() cho từng record, ghi ra JSONL sạch."""
    total = kept = 0
    with input_path.open("r", encoding="utf-8") as in_f, output_path.open("w", encoding="utf-8") as out_f:
        for line in in_f:
            line = line.strip()
            if not line: continue
            total += 1
            record = json.loads(line)
            record["text"] = clean_wikitext(record.get("text", ""))
            out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
            kept += 1
            if kept % 500 == 0: logger.info("Processed {} articles (kept {} so far)...", total, kept)
    logger.info("Done. Total read : {} | Kept : {} | Output: {}", total, kept, output_path)


def convert_jsonl_to_parquet(input_path: Path, output_parquet: Path) -> None:
    """Chuyển đổi JSONL sạch sang định dạng Parquet."""
    logger.info("Converting JSONL to Parquet...")
    records = []
    with input_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                try: records.append(json.loads(line))
                except json.JSONDecodeError: continue
    if not records: return
    ds = Dataset.from_list(records)
    ds.to_parquet(output_parquet)
    def format_size(size): return f"{size/1e6:.2f} MB" if size > 1e6 else f"{size/1e3:.2f} KB"
    logger.info("Saved : {} ({})", output_parquet, format_size(os.path.getsize(output_parquet)))

In [ ]:
# --- Chạy pipeline làm sạch Wikipedia ---
INPUT_JSONL = Path("data/raws/vi_wiki_articles.jsonl")
CLEAN_JSONL = Path("data/raws/vi_wiki_articles_clean.jsonl")
WIKI_TRAIN_DIR = Path("data/train")

if INPUT_JSONL.exists():
    process(INPUT_JSONL, CLEAN_JSONL)
    WIKI_TRAIN_DIR.mkdir(parents=True, exist_ok=True)
    out_parquet = WIKI_TRAIN_DIR / "vi_wiki_articles_clean.parquet"
    convert_jsonl_to_parquet(CLEAN_JSONL, out_parquet)
else:
    logger.error("Ch\u01b0a th\u1ea5y file raw vi_wiki_articles.jsonl. Vui l\u00f2ng ch\u1ea1y cell crawl Wiki tr\u01b0\u1edbc.")


---
## Phần 4: Loại bỏ nội dung trùng lặp (Deduplicate)

Toàn bộ logic từ `deduplicate.py` thực hiện Exact-match Hash qua SHA-256 để tạo corpus sạch chuẩn bị train.

**Chiến lược dedup 2 tầng:**
1. **Doc-level dedup (raw)**: Hash SHA-256 toàn bộ document gốc → loại bỏ bản sao chính xác.
2. **Paragraph-level dedup**: Chia document thành các đoạn, loại bỏ đoạn trùng lặp giữa các document (chỉ áp dụng cho BKAI + Wiki).
3. **Doc-level dedup (final)**: Hash document sau khi đã lọc đoạn → loại bỏ các document trở nên giống nhau.
4. **Lọc độ dài**: Bỏ document quá ngắn (`< MIN_DOC_CHARS`).

In [ ]:
# --- Cấu hình Deduplication ---

class cfg:
    DEDUP_DIR = "data/train/deduped"
    RAW_DATASETS = [
        "data/train/bkai_train.parquet",
        "data/train/vi_wiki_articles_clean.parquet",
    ]

DEDUP_DIR = Path(cfg.DEDUP_DIR)
PARA_STEMS = {"bkai_train", "vi_wiki_articles_clean"}
MIN_PARA_CHARS = 50
MIN_DOC_CHARS = 20
BATCH_SIZE = 50_000

In [ ]:
# --- Hàm tiện ích: normalize, hash SHA-256, dedup các đoạn văn ---

def normalize_text(text: str) -> str:
    """Chuẩn hóa Unicode NFC để hash nhất quán."""
    if text is None:
        return ""
    return unicodedata.normalize("NFC", text)


def sha_bytes(text):
    """Trả về SHA-256 digest (bytes) của text đã normalize."""
    return hashlib.sha256(normalize_text(text).encode("utf-8")).digest()


def dedup_paragraphs(text, seen_paras):
    """Loại bỏ các đoạn văn trùng lặp (>= MIN_PARA_CHARS) trong document."""
    kept = []
    for para in text.split("\n\n"):
        para = para.strip()
        if not para:
            continue
        if len(para) < MIN_PARA_CHARS:
            kept.append(para)
            continue
        h = sha_bytes(para)
        if h in seen_paras:
            continue
        seen_paras.add(h)
        kept.append(para)
    return "\n\n".join(kept).strip()

In [ ]:
# --- Hàm ghi kết quả ra Parquet theo batch ---

def flush_rows(writer, out_path, rows):
    """Ghi batch rows vào Parquet file, khởi tạo writer nếu chưa có."""
    if not rows:
        return writer
    table = pa.table({"text": rows})
    if writer is None:
        writer = pq.ParquetWriter(out_path, table.schema, compression="snappy")
    writer.write_table(table)
    rows.clear()
    return writer

In [ ]:
# --- Hàm dedup chính: xử lý toàn bộ datasets ---

def dedup_all():
    """Chạy toàn bộ pipeline deduplication cho các dataset được cấu hình."""
    os.makedirs(DEDUP_DIR, exist_ok=True)
    seen_docs_raw, seen_docs_final = set(), set()
    outputs, report = [], {"sources": {}}
    total_original, total_kept = 0, 0

    for path in cfg.RAW_DATASETS:
        if not os.path.exists(path):
            logger.warning(f"Skipping (not found): {path}")
            continue

        name, filename = Path(path).stem, Path(path).name
        parquet = pq.ParquetFile(path)
        original_docs = parquet.metadata.num_rows

        out_path = DEDUP_DIR / filename
        if out_path.exists():
            out_path.unlink()

        para_seen = set()
        use_para = name in PARA_STEMS
        writer, out_rows = None, []
        stats = {
            "original_docs": original_docs,
            "deduped_docs": 0,
            "removed_by_doc_dedup": 0,
            "removed_by_paragraph_dedup": 0,
            "removed_as_too_short": 0,
        }

        for batch in tqdm(parquet.iter_batches(batch_size=BATCH_SIZE, columns=["text"]), desc=f"Dedup {name}"):
            for text in batch.column(0).to_pylist():
                total_original += 1
                text = text or ""
                if not text:
                    stats["removed_as_too_short"] += 1
                    continue

                raw_hash = sha_bytes(text)
                if raw_hash in seen_docs_raw:
                    stats["removed_by_doc_dedup"] += 1
                    continue
                seen_docs_raw.add(raw_hash)

                new_text = dedup_paragraphs(text, para_seen) if use_para else text.strip()
                if new_text != text.strip() and use_para:
                    stats["removed_by_paragraph_dedup"] += 1

                if len(new_text) < MIN_DOC_CHARS:
                    stats["removed_as_too_short"] += 1
                    continue

                final_hash = sha_bytes(new_text)
                if final_hash in seen_docs_final:
                    stats["removed_by_doc_dedup"] += 1
                    continue
                seen_docs_final.add(final_hash)

                out_rows.append(new_text)
                stats["deduped_docs"] += 1
                total_kept += 1
                if len(out_rows) >= BATCH_SIZE:
                    writer = flush_rows(writer, out_path, out_rows)

        writer = flush_rows(writer, out_path, out_rows)
        if writer is not None:
            writer.close()
        else:
            pq.write_table(pa.table({"text": []}), out_path, compression="snappy")

        stats["removed_docs"] = stats["original_docs"] - stats["deduped_docs"]
        stats["duplicate_rate"] = round(stats["removed_docs"] / stats["original_docs"], 4) if stats["original_docs"] else 0.0
        stats["output_path"] = str(out_path)
        report["sources"][name] = stats
        outputs.append({"name": name, "path": str(out_path), "rows": stats["deduped_docs"]})

        logger.info("{} {} -> {} (-{})", name, f"{stats['original_docs']:,}", f"{stats['deduped_docs']:,}", f"{stats['removed_docs']:,}")

    report["total_original_docs"] = total_original
    report["total_deduped_docs"] = total_kept
    report["total_removed_docs"] = total_original - total_kept
    report["duplicate_rate"] = round((total_original - total_kept) / total_original, 4) if total_original else 0.0
    return outputs, report